In [ ]:
import os, sys
import pandas as pd
import numpy as np
from pathlib import Path
from multiprocessing import get_context

PROJECT_ROOT = Path.cwd().parents[0]
sys.path.insert(0, str(PROJECT_ROOT))

from greeks_calculator import compute_greeks

In [ ]:
df = pd.read_csv("../data/vanilla_convertibles_data.csv")
print(f"Rows: {len(df):,}")
df.head()

In [ ]:
def run_greeks_multi_cpu(
    df,
    n_procs=None,
    chunk_size=15_000,
    out_dir="greeks_output",
):
    out_dir = os.path.abspath(out_dir)
    os.makedirs(out_dir, exist_ok=True)

    if n_procs is None:
        n_procs = max(1, os.cpu_count() - 2)

    splits = np.array_split(df, n_procs)

    jobs = []
    for wid, split_df in enumerate(splits):
        jobs.append((split_df.reset_index(drop=True), chunk_size, out_dir, wid))

    ctx = get_context("spawn")
    with ctx.Pool(processes=n_procs) as pool:
        pool.starmap(compute_greeks, jobs)

In [ ]:
run_greeks_multi_cpu(df, n_procs=7, out_dir="greeks_output")

In [ ]:
out_dir = "greeks_output"
parts = []
for wid in range(7):
    chunk_id = 0
    while True:
        path = os.path.join(out_dir, f"w{wid}_greeks_chunk{chunk_id}.csv")
        if not os.path.exists(path):
            break
        parts.append(pd.read_csv(path))
        chunk_id += 1

greeks_df = pd.concat(parts, ignore_index=True)
print(f"Greeks rows: {len(greeks_df):,}")
greeks_df.describe()

In [ ]:
df["delta"] = greeks_df["delta"].values
df["gamma"] = greeks_df["gamma"].values
df["vega"]  = greeks_df["vega"].values

print(f"NaN count: delta={df['delta'].isna().sum()}, gamma={df['gamma'].isna().sum()}, vega={df['vega'].isna().sum()}")
df[["S", "bs_vol", "maturity_years", "price_convertible", "delta", "gamma", "vega"]].describe()

In [ ]:
df.to_csv("../data/vanilla_convertibles_data_greeks.csv", index=False)
print("Saved to data/vanilla_convertibles_data_greeks.csv")